# Video Splitter — MediaPipe Human Detection

Split `video.mp4` into sub-chunks based on human presence and action continuity.

**Pipeline:**
1. Scan every Nth frame with MediaPipe Pose to detect human presence & landmark positions
2. Group consecutive human-present frames into segments
3. Within each segment, detect action jumps (large landmark displacement) to split into chunks
4. Export chunks with audio to `subclips/` and save metadata to `chunks.json`

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import json
import os
from moviepy import VideoFileClip

In [2]:
# ── Config ──────────────────────────────────────────
VIDEO_PATH = "video.mp4"
OUTPUT_DIR = "subclips"
JSON_PATH = "chunks.json"

SAMPLE_EVERY = 5          # analyse every Nth frame (speed vs accuracy)
MIN_CHUNK_FRAMES = 50     # minimum frames to keep a chunk (skip tiny segments)
JUMP_THRESHOLD = 0.15     # normalised landmark displacement to trigger new chunk
NO_HUMAN_GAP = 25         # consecutive no-human frames to end a segment
CENTER_MARGIN = 0.15      # person must be within 0.5 ± this margin horizontally (0.35–0.65)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 1 — Scan video with MediaPipe Pose

In [3]:
mp_pose = mp.solutions.pose
mp_face = mp.solutions.face_detection

# Use moviepy to read frames (supports AV1 codec, unlike OpenCV)
video_clip = VideoFileClip(VIDEO_PATH)
fps = video_clip.fps
total_frames = int(video_clip.duration * fps)
print(f"Video: {total_frames} frames, {fps} FPS, ~{video_clip.duration:.0f}s")

# Store per-frame info: (frame_idx, human_detected, landmark_center)
frame_data = []
multi_person_skipped = 0
off_center_skipped = 0

with mp_face.FaceDetection(
    model_selection=1,            # full-range model (better for far faces)
    min_detection_confidence=0.5,
) as face_det, mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as pose:
    for frame_idx, frame in enumerate(video_clip.iter_frames(dtype="uint8")):
        if frame_idx % SAMPLE_EVERY == 0:
            # Step A: count faces — skip if multiple people
            face_results = face_det.process(frame)
            n_faces = len(face_results.detections) if face_results.detections else 0

            if n_faces > 1:
                frame_data.append((frame_idx, False, None))
                multi_person_skipped += 1
            else:
                # Step B: run pose on single-person (or empty) frames
                results = pose.process(frame)

                if results.pose_landmarks:
                    lm = results.pose_landmarks.landmark
                    xs = [lm[0].x, lm[23].x, lm[24].x]
                    ys = [lm[0].y, lm[23].y, lm[24].y]
                    cx = np.mean(xs)
                    cy = np.mean(ys)

                    # Step C: person must be horizontally centered
                    if abs(cx - 0.5) > CENTER_MARGIN:
                        frame_data.append((frame_idx, False, None))
                        off_center_skipped += 1
                    else:
                        frame_data.append((frame_idx, True, (cx, cy)))
                else:
                    frame_data.append((frame_idx, False, None))

        if frame_idx % 5000 == 0 and frame_idx > 0:
            print(f"  scanned {frame_idx}/{total_frames} frames...")

video_clip.close()
human_count = sum(1 for _, h, _ in frame_data if h)
no_human = len(frame_data) - human_count - multi_person_skipped - off_center_skipped
print(f"\nScan complete. {len(frame_data)} sampled frames:")
print(f"  - {human_count} with single centered human")
print(f"  - {multi_person_skipped} skipped (multiple people)")
print(f"  - {off_center_skipped} skipped (person not centered)")
print(f"  - {no_human} no human detected")

Video: 94330 frames, 25.0 FPS, ~3773s


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1771959510.294758  152686 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
W0000 00:00:1771959510.719778  152695 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1771959510.733660  152695 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1771959510.743182  152695 landmark_projection_

  scanned 5000/94330 frames...
  scanned 10000/94330 frames...
  scanned 15000/94330 frames...
  scanned 20000/94330 frames...
  scanned 25000/94330 frames...
  scanned 30000/94330 frames...
  scanned 35000/94330 frames...
  scanned 40000/94330 frames...
  scanned 45000/94330 frames...
  scanned 50000/94330 frames...
  scanned 55000/94330 frames...
  scanned 60000/94330 frames...
  scanned 65000/94330 frames...
  scanned 70000/94330 frames...
  scanned 75000/94330 frames...
  scanned 80000/94330 frames...
  scanned 85000/94330 frames...
  scanned 90000/94330 frames...

Scan complete. 18866 sampled frames:
  - 18133 with single centered human
  - 10 skipped (multiple people)
  - 29 skipped (person not centered)
  - 694 no human detected


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file video.mp4, 6220800 bytes wanted but 0 bytes read at frame index 94329 (out of a total 94330 frames), at time 3773.16/3773.22 sec. Using the last valid frame instead.
  warnings.warn(


## Step 2 — Find human-present segments & detect action jumps

In [4]:
def find_chunks(frame_data, no_human_gap, jump_threshold, min_chunk_frames, sample_every):
    """Split frame_data into chunks based on human presence and action continuity."""
    chunks = []
    current_start = None
    prev_center = None
    gap_count = 0

    for fidx, has_human, center in frame_data:
        if has_human:
            if current_start is None:
                # new segment begins
                current_start = fidx
                prev_center = center
                gap_count = 0
                continue

            gap_count = 0

            # check for action jump
            if prev_center is not None:
                dist = np.sqrt((center[0] - prev_center[0])**2 +
                               (center[1] - prev_center[1])**2)
                if dist > jump_threshold:
                    # action jump -> end current chunk, start new
                    chunk_end = fidx - 1
                    if (chunk_end - current_start) >= min_chunk_frames:
                        chunks.append((current_start, chunk_end))
                    current_start = fidx

            prev_center = center

        else:
            if current_start is not None:
                gap_count += sample_every
                if gap_count >= no_human_gap:
                    # long enough gap without human -> end segment
                    chunk_end = fidx - gap_count
                    if (chunk_end - current_start) >= min_chunk_frames:
                        chunks.append((current_start, chunk_end))
                    current_start = None
                    prev_center = None
                    gap_count = 0

    # close last chunk
    if current_start is not None:
        last_frame = frame_data[-1][0]
        if (last_frame - current_start) >= min_chunk_frames:
            chunks.append((current_start, last_frame))

    return chunks


chunks = find_chunks(frame_data, NO_HUMAN_GAP, JUMP_THRESHOLD, MIN_CHUNK_FRAMES, SAMPLE_EVERY)
print(f"Found {len(chunks)} chunks:")
for i, (s, e) in enumerate(chunks):
    dur = (e - s) / fps
    print(f"  chunk {i:03d}: frames {s} - {e}  ({dur:.1f}s)")

Found 39 chunks:
  chunk 000: frames 0 - 224  (9.0s)
  chunk 001: frames 225 - 309  (3.4s)
  chunk 002: frames 310 - 379  (2.8s)
  chunk 003: frames 380 - 459  (3.2s)
  chunk 004: frames 460 - 609  (6.0s)
  chunk 005: frames 610 - 1194  (23.4s)
  chunk 006: frames 1195 - 1435  (9.6s)
  chunk 007: frames 1585 - 5530  (157.8s)
  chunk 008: frames 5645 - 5774  (5.2s)
  chunk 009: frames 5900 - 14515  (344.6s)
  chunk 010: frames 14640 - 17185  (101.8s)
  chunk 011: frames 17310 - 18794  (59.4s)
  chunk 012: frames 18910 - 20610  (68.0s)
  chunk 013: frames 20735 - 22185  (58.0s)
  chunk 014: frames 22310 - 22599  (11.6s)
  chunk 015: frames 22715 - 23584  (34.8s)
  chunk 016: frames 23705 - 37659  (558.2s)
  chunk 017: frames 37780 - 45354  (303.0s)
  chunk 018: frames 45465 - 46855  (55.6s)
  chunk 019: frames 46985 - 53130  (245.8s)
  chunk 020: frames 53260 - 56184  (117.0s)
  chunk 021: frames 56310 - 57644  (53.4s)
  chunk 022: frames 57770 - 59075  (52.2s)
  chunk 023: frames 59205 

## Step 3 — Export subclips with audio & save JSON

In [5]:
!pwd

/home/dipcik/phdprojects/flame-head-tracker/get_video


In [6]:
# Save chunks.json
chunks_meta = []
for i, (start_frame, end_frame) in enumerate(chunks):
    chunks_meta.append({
        "chunk_id": i,
        "start_frame": int(start_frame),
        "end_frame": int(end_frame),
        "start_time_s": round(start_frame / fps, 3),
        "end_time_s": round(end_frame / fps, 3),
        "duration_s": round((end_frame - start_frame) / fps, 3),
        "filename": f"chunk_{i:03d}.mp4"
    })

with open(JSON_PATH, "w") as f:
    json.dump(chunks_meta, f, indent=2)

print(f"Saved {JSON_PATH} with {len(chunks_meta)} entries.")

Saved chunks.json with 39 entries.


In [7]:
# Export subclips with audio using moviepy
video = VideoFileClip(VIDEO_PATH)

for meta in chunks_meta:
    t_start = meta["start_time_s"]
    t_end = meta["end_time_s"]
    out_path = os.path.join(OUTPUT_DIR, meta["filename"])

    subclip = video.subclipped(t_start, t_end)
    subclip.write_videofile(
        out_path,
        codec="libx264",
        audio_codec="aac",
        logger="bar",
    )
    print(f"  >> {meta['filename']} ({meta['duration_s']:.1f}s)")

video.close()
print(f"\nDone! {len(chunks_meta)} subclips saved to {OUTPUT_DIR}/")

MoviePy - Building video subclips/chunk_000.mp4.
MoviePy - Writing audio in chunk_000TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips/chunk_000.mp4



MoviePy - Done !
MoviePy - video ready subclips/chunk_000.mp4
  >> chunk_000.mp4 (9.0s)
MoviePy - Building video subclips/chunk_001.mp4.
MoviePy - Writing audio in chunk_001TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips/chunk_001.mp4



MoviePy - Done !
MoviePy - video ready subclips/chunk_001.mp4
  >> chunk_001.mp4 (3.4s)
MoviePy - Building video subclips/chunk_002.mp4.
MoviePy - Writing audio in chunk_002TEMP_MPY_wvf_snd.mp4


...

MoviePy - Done !
MoviePy - video ready subclips/chunk_036.mp4
  >> chunk_036.mp4 (54.8s)
MoviePy - Building video subclips/chunk_037.mp4.
MoviePy - Writing audio in chunk_037TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips/chunk_037.mp4



MoviePy - Done !
MoviePy - video ready subclips/chunk_037.mp4
  >> chunk_037.mp4 (74.0s)
MoviePy - Building video subclips/chunk_038.mp4.
MoviePy - Writing audio in chunk_038TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips/chunk_038.mp4



MoviePy - Done !
MoviePy - video ready subclips/chunk_038.mp4
  >> chunk_038.mp4 (186.4s)

Done! 39 subclips saved to subclips/


## Summary

In [8]:
total_dur = sum(m["duration_s"] for m in chunks_meta)
print(f"Total chunks: {len(chunks_meta)}")
print(f"Total duration kept: {total_dur:.1f}s ({total_dur/60:.1f}min)")
print(f"Original duration: {total_frames/fps:.1f}s ({total_frames/fps/60:.1f}min)")
print(f"Removed: {(1 - total_dur/(total_frames/fps))*100:.1f}% of video (no human / too short)")

Total chunks: 39
Total duration kept: 3609.6s (60.2min)
Original duration: 3773.2s (62.9min)
Removed: 4.3% of video (no human / too short)


In [9]:
# Check Video Chunks frame by frame, find the videos with no person detected in any frame
import glob

chunk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "chunk_*.mp4")))
print(f"Checking {len(chunk_files)} chunks for missing person detections...\n")

CHECK_EVERY = 2  # check every Nth frame within each chunk

flagged = []

with mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as pose:
    for chunk_path in chunk_files:
        fname = os.path.basename(chunk_path)
        clip = VideoFileClip(chunk_path)
        n_frames = int(clip.duration * clip.fps)
        no_person_frames = []

        for fidx, frame in enumerate(clip.iter_frames(dtype="uint8")):
            if fidx % CHECK_EVERY == 0:
                results = pose.process(frame)
                if not results.pose_landmarks:
                    no_person_frames.append(fidx)

        clip.close()
        checked = n_frames // CHECK_EVERY
        ratio = len(no_person_frames) / max(checked, 1)

        if no_person_frames:
            flagged.append({
                "filename": fname,
                "total_frames": n_frames,
                "checked_frames": checked,
                "no_person_frames": len(no_person_frames),
                "ratio": round(ratio, 3),
                "sample_missing": no_person_frames[:10],  # first 10 examples
            })
            print(f"  {fname}: {len(no_person_frames)}/{checked} frames no person ({ratio*100:.1f}%)")
        else:
            print(f"  {fname}: OK")

print(f"\n{'='*50}")
print(f"Flagged: {len(flagged)}/{len(chunk_files)} chunks have frames without person")
for f in flagged:
    print(f"  {f['filename']}: {f['no_person_frames']} missing in {f['checked_frames']} checked ({f['ratio']*100:.1f}%)")

Checking 33 chunks for missing person detections...



W0000 00:00:1771960528.047344  160910 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1771960528.060965  160910 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips/chunk_007.mp4, 6220800 bytes wanted but 0 bytes read at frame index 3944 (out of a total 3945 frames), at time 157.76/157.80 sec. Using the last valid frame inst

  chunk_007.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips/chunk_008.mp4, 6220800 bytes wanted but 0 bytes read at frame index 128 (out of a total 129 frames), at time 5.12/5.16 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_008.mp4: OK
  chunk_009.mp4: 2/4307 frames no person (0.0%)


...

  chunk_026.mp4: OK
  chunk_027.mp4: OK
  chunk_028.mp4: 4/1842 frames no person (0.2%)
  chunk_029.mp4: 2/329 frames no person (0.6%)
  chunk_030.mp4: OK
  chunk_031.mp4: OK
  chunk_032.mp4: 8/647 frames no person (1.2%)
  chunk_033.mp4: 7/994 frames no person (0.7%)
  chunk_034.mp4: 8/842 frames no person (1.0%)


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips/chunk_035.mp4, 6220800 bytes wanted but 0 bytes read at frame index 1654 (out of a total 1655 frames), at time 66.16/66.20 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_035.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips/chunk_036.mp4, 6220800 bytes wanted but 0 bytes read at frame index 1368 (out of a total 1369 frames), at time 54.72/54.76 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_036.mp4: 6/684 frames no person (0.9%)
  chunk_037.mp4: OK
  chunk_038.mp4: OK
  chunk_039.mp4: OK

Flagged: 13/33 chunks have frames without person
  chunk_009.mp4: 2 missing in 4307 checked (0.0%)
  chunk_010.mp4: 1 missing in 1272 checked (0.1%)
  chunk_011.mp4: 5 missing in 742 checked (0.7%)
  chunk_014.mp4: 7 missing in 144 checked (4.9%)
  chunk_016.mp4: 4 missing in 6977 checked (0.1%)
  chunk_021.mp4: 1 missing in 667 checked (0.1%)
  chunk_025.mp4: 1 missing in 942 checked (0.1%)
  chunk_028.mp4: 4 missing in 1842 checked (0.2%)
  chunk_029.mp4: 2 missing in 329 checked (0.6%)
  chunk_032.mp4: 8 missing in 647 checked (1.2%)
  chunk_033.mp4: 7 missing in 994 checked (0.7%)
  chunk_034.mp4: 8 missing in 842 checked (1.0%)
  chunk_036.mp4: 6 missing in 684 checked (0.9%)


In [ ]:
# For each flagged chunk, find how many consecutive no-person frames are at the END,
# trim only those, and overwrite the file. Update chunks.json accordingly.

trimmed_count = 0
for f in flagged:
    fname = f["filename"]
    missing_frames = sorted(f["sample_missing"])  # sample_missing has first 10, need full list
    chunk_path = os.path.join(OUTPUT_DIR, fname)

    # Re-scan the tail of the chunk to find consecutive trailing no-person frames
    clip = VideoFileClip(chunk_path)
    n_frames = int(clip.duration * clip.fps)

    # Scan last portion of the video backwards to find trailing no-person count
    trailing_count = 0
    frames_list = list(clip.iter_frames(dtype="uint8"))
    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=1,
        min_detection_confidence=0.5,
    ) as pose_check:
        for i in range(len(frames_list) - 1, -1, -1):
            results = pose_check.process(frames_list[i])
            if results.pose_landmarks:
                break
            trailing_count += 1

    if trailing_count == 0:
        print(f"  {fname}: no trailing no-person frames, skipping (missing frames are mid-video)")
        clip.close()
        continue

    trim_seconds = trailing_count / clip.fps
    new_end = clip.duration - trim_seconds

    if new_end <= 0.5:
        print(f"  SKIP {fname}: would trim entire video")
        clip.close()
        continue

    tmp_path = chunk_path.replace(".mp4", "_trimmed.mp4")
    trimmed = clip.subclipped(0, new_end)
    trimmed.write_videofile(tmp_path, codec="libx264", audio_codec="aac", logger="bar")
    clip.close()

    os.replace(tmp_path, chunk_path)
    trimmed_count += 1
    print(f"  {fname}: trimmed {trailing_count} trailing frames ({trim_seconds:.2f}s) -> {new_end:.2f}s")

# Update chunks.json with trimmed durations
with open(JSON_PATH, "r") as jf:
    chunks_meta = json.load(jf)

for meta in chunks_meta:
    # Re-read actual duration from file
    fpath = os.path.join(OUTPUT_DIR, meta["filename"])
    if os.path.exists(fpath):
        c = VideoFileClip(fpath)
        meta["end_time_s"] = round(meta["start_time_s"] + c.duration, 3)
        meta["duration_s"] = round(c.duration, 3)
        meta["end_frame"] = int(meta["start_frame"] + c.duration * fps)
        c.close()

with open(JSON_PATH, "w") as jf:
    json.dump(chunks_meta, jf, indent=2)

print(f"\nDone! Trimmed {trimmed_count}/{len(flagged)} flagged chunks. Updated {JSON_PATH}.")